In [1]:
import hashlib
import os
import platform
import uuid

# 修改为自己的学号
STUDENT_ID = 20244081127

CONDA_ENV = os.environ.get("CONDA_DEFAULT_ENV", "base")
print(f"[Conda 环境] {CONDA_ENV}")

MACHINE_FP = hashlib.sha256(
    f"{platform.node()}|{platform.machine()}|{platform.system()}|{uuid.getnode()}".encode()
).hexdigest()[:16]
RUN_ID = f"{STUDENT_ID}-{MACHINE_FP}"
print(f"[学号]       {STUDENT_ID}")
print(f"[机器指纹] {MACHINE_FP}")
print(f"[RUN_ID]   {RUN_ID}")

[Conda 环境] exp_env
[学号]       20244081127
[机器指纹] 129b330c7033e382
[RUN_ID]   20244081127-129b330c7033e382


## 字符串的乐趣

In [2]:
# ========== 任务1 判断回文 ==========
def is_palindrome(s):
    # 去掉空格，全部转小写
    raw = s.replace(" ", "").lower()
    return raw == raw[::-1]

print(is_palindrome("1234321"))
print(is_palindrome("tacocat"))
print(is_palindrome("T A C O cat"))

True
True
True


In [3]:
# ========== 任务2 判断字母顺序 ==========
def is_abecedarian(s):
    # 去掉空格，全部转小写
    raw = s.replace(" ", "").lower()
    for i in range(len(raw)-1):
        if raw[i] > raw[i+1]:
            return False
    return True

print(is_abecedarian("adder"))
print(is_abecedarian("abcd efgh xyz"))
print(is_abecedarian("dog"))

True
True
False


In [4]:
# ========== 任务3 删除元音 ==========
def remove_vowels(s):
    vowels = {'a','e','i','o','u','A','E','I','O','U'}
    res = ""
    for char in s:
        if char not in vowels:
            res += char
    return res

print(remove_vowels('cat'))   
print(remove_vowels('audio'))  
print(remove_vowels('AUDIO'))  

ct
d
D


## 列表的乐趣

In [5]:
# ========== 任务1 反转列表 ==========
def list_reverse(lst):
    if not isinstance(lst, list):
        raise TypeError("输入必须是列表")
    return lst[::-1]

print(list_reverse([1,2,3]))
try:
    print(list_reverse(123))
except TypeError as e:
    print(f"list_reverse(123) 捕获到异常：{e}")

[3, 2, 1]
list_reverse(123) 捕获到异常：输入必须是列表


In [6]:
# ========== 任务2 判断序列是否升序 ==========
def is_sorted(seq):
    for i in range(len(seq)-1):
        if seq[i] > seq[i+1]:
            return False
    return True

print(is_sorted([1,2,3,4]))
print(is_sorted([1,3,2]))
print(is_sorted("abcde"))

True
False
True


## 作为向量的元组

In [7]:
# ========== 任务1 向量数乘 ==========
def vec_scalar_mult(t, s):
    # 检查t是否为元组
    if not isinstance(t, tuple):
        raise TypeError("第一个参数必须是元组")
    # 检查s是否数字
    if not isinstance(s, (int, float)):
        raise TypeError("第二个参数必须是整数或浮点数")
    # 检查元组内元素是int/float
    for elem in t:
        if not isinstance(elem, (int, float)):
            raise TypeError("元组内元素必须是整数或浮点数")
    new_tuple = tuple(x * s for x in t)
    return new_tuple

print(vec_scalar_mult( (1,2,3), 2 ))

(2, 4, 6)


In [8]:
# ========== 任务2 向量内积 ==========
def vec_inner_product(v1, v2):
    if not isinstance(v1, tuple) or not isinstance(v2, tuple):
        raise TypeError("输入必须是元组")
    if len(v1) != len(v2):
        raise ValueError("两个向量长度不一致")
    total = 0.0
    for a,b in zip(v1, v2):
        # 校验向量元素类型
        if not isinstance(a,(int,float)) or not isinstance(b,(int,float)):
            raise TypeError("向量元素必须是整数或浮点数")
        total += a * b
    return total

print(vec_inner_product( (1,2), (3,4) ))

11.0


In [9]:
# ========== 任务3 检查有效矩阵 ==========
def check_valid_mx(mx):
    # 外层必须是元组
    if not isinstance(mx, tuple):
        return False
    # 遍历外层每一行
    row_len = None
    for row in mx:
        # 每一行必须是元组
        if not isinstance(row, tuple):
            return False
        # 记录第一行长度，后续行必须和它一样
        if row_len is None:
            row_len = len(row)
        else:
            if len(row) != row_len:
                return False
        # 检查该行所有元素是int/float
        for elem in row:
            if not isinstance(elem, (int, float)):
                return False
    return True

mx_test1 = ((1,2),(3,4))
print(check_valid_mx(mx_test1))
mx_test2 = (('a',2),(4,5))
print(check_valid_mx(mx_test2))

True
False


In [10]:
# ========== 任务4 矩阵乘向量 ==========
def mx_vec_mult(mx, vec):
    # 先用check_valid_mx校验矩阵
    if not check_valid_mx(mx):
        raise TypeError("矩阵格式非法")
    if not isinstance(vec, tuple):
        raise TypeError("向量必须是元组")
    # 获取矩阵列数 = 向量长度
    cols = len(mx[0]) if mx else 0
    if len(vec) != cols:
        raise ValueError("矩阵列数和向量长度不匹配")
    res = []
    for row in mx:
        # 每行和vec做内积
        sum_val = 0.0
        for a,b in zip(row, vec):
            sum_val += a * b
        res.append(sum_val)
    return tuple(res)

print(mx_vec_mult( ((1,2),(3,4)), (5,6) ))

(17.0, 39.0)


## 更多向量乐趣

In [11]:
# ========== 任务1 类Vector ==========
class Vector:
    def __init__(self, dim, elements=None):
        # 校验维度
        if not isinstance(dim, int):
            raise TypeError("维度必须是整数")
        if dim < 0:
            raise ValueError("维度不能为负数")
        
        # 默认元素：全部0
        if elements is None:
            elements = [0]*dim
        
        # 转成元组保存，保证不可变
        ele_tuple = tuple(elements)
        if len(ele_tuple) != dim:
            raise ValueError("元素数量必须和维度一致")
        for num in ele_tuple:
            if not isinstance(num, (int, float)):
                raise TypeError("元素只能是整数或浮点数")
        
        # 实例属性
        self.dim = dim
        self.elements = ele_tuple

    # 自定义打印输出
    def __repr__(self):
        return f"Vector({self.dim}, {self.elements})"

    # ========== 任务3 比较运算符 ==========
    def __eq__(self, other):
        if not isinstance(other, Vector):
            return False
        if self.dim != other.dim:
            raise ValueError("维度不同，无法比较")
        return self.elements == other.elements

    def __lt__(self, other):
        if not isinstance(other, Vector):
            raise TypeError("只能和Vector实例比较")
        if self.dim != other.dim:
            raise ValueError("维度不同，无法比较")
        return self.elements < other.elements

    def __le__(self, other):
        if not isinstance(other, Vector):
            raise TypeError("只能和Vector实例比较")
        if self.dim != other.dim:
            raise ValueError("维度不同，无法比较")
        return self.elements <= other.elements

    def __gt__(self, other):
        if not isinstance(other, Vector):
            raise TypeError("只能和Vector实例比较")
        if self.dim != other.dim:
            raise ValueError("维度不同，无法比较")
        return self.elements > other.elements

    def __ge__(self, other):
        if not isinstance(other, Vector):
            raise TypeError("只能和Vector实例比较")
        if self.dim != other.dim:
            raise ValueError("维度不同，无法比较")
        return self.elements >= other.elements

    # ========== 任务4 dot内积方法 ==========
    def dot(self, other):
        if not isinstance(other, Vector):
            raise TypeError("参数必须是Vector对象")
        if self.dim != other.dim:
            raise ValueError("两个向量维度不一致，无法计算内积")
        total = 0.0
        for a,b in zip(self.elements, other.elements):
            total += a * b
        return total

    # ========== 任务5 乘法运算符 ==========
    def __mul__(self, other):
        # 向量 * 向量：逐元素相乘
        if isinstance(other, Vector):
            if self.dim != other.dim:
                raise ValueError("向量维度不一致，无法逐元素相乘")
            new_ele = tuple(a*b for a,b in zip(self.elements, other.elements))
            return Vector(self.dim, new_ele)
        # 向量 * 标量
        elif isinstance(other, (int, float)):
            new_ele = tuple(x * other for x in self.elements)
            return Vector(self.dim, new_ele)
        else:
            raise TypeError("不支持的乘法类型")
    
    # 标量左乘
    def __rmul__(self, other):
        return self * other

In [12]:
# 测试
v1 = Vector(2, (2,4))
v2 = Vector(2, (2,5))
print(v1 < v2)

# 内积dot
v3 = Vector(2,(1,2))
v4 = Vector(2,(3,4))
print(v3.dot(v4))

# 标量乘法
print(2 * v3)
print(v3 * 2)

# 向量逐元素相乘
v_a = Vector(2,(2,3))
v_b = Vector(2,(4,5))
v_c = v_a * v_b
print(v_c)

True
11.0
Vector(2, (2, 4))
Vector(2, (2, 4))
Vector(2, (8, 15))


维度dim和元素elements应当作为实例属性。  
理由：  
- 每个Vector实例代表一个独立向量，不同向量的维度、元素可以完全不同。
- 类属性属于整个类，所有实例共用同一份数据，不适合存储每个向量独有的维度和分量。

例如向量Vector(2,(1,2))和Vector(3,(1,2,3))，各自拥有独立的dim与elements，因此定义为实例属性。